In [0]:
from pyspark.sql.functions import row_number, floor, col, avg, abs, round
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text(
    "silver_catalog",
     "dbr_dev")
dbutils.widgets.text(
    "silver_schema",
    "artemzharkov10_silver"
)

dbutils.widgets.text(
    "gold_catalog",
    "dbr_dev")
dbutils.widgets.text(
    "gold_schema",
    "artemzharkov10_gold"
)

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")


In [0]:

df_bitcoin = spark.read.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.finance_bitcoin_silver")

df_bitcoin = df_bitcoin.orderBy(col("TradeDate"))

window_spec = Window.orderBy("TradeDate")
df_numbered = df_bitcoin.withColumn("row_number", row_number().over(window_spec))


df_grouped = df_numbered.withColumn("GroupId",floor((col("row_number")-1)/14))

by_group = Window.partitionBy("GroupID")
df_with_avg = df_grouped.withColumn("GroupAvgPrice", round(avg("Close").over(by_group), 2))

df_deviation = df_with_avg.withColumn(
    "DeviationPct",
    round(((col("Close") - col("GroupAvgPrice")) / col("GroupAvgPrice")) * 100, 2)
)

DEVIATION_LEVEL = 29.0
df_gold_anomalies = df_deviation.filter((col("DeviationPct") >= DEVIATION_LEVEL) | (col("DeviationPct") <= -DEVIATION_LEVEL))

df_gold_final = df_gold_anomalies.select(
    col("TradeDate").alias("AnomalyDate"),
    col("Close").alias("PriceOnAnomalyDay"),
    col("GroupAvgPrice").alias("BlockAveragePrice"),
    col("DeviationPct").alias("DeviationPercentage")
)

df_gold_final.write.mode("overwrite").saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.finance_bitcoin_anomaly_gold")
# display(df_gold_final.orderBy(col("DeviationPercentage").asc()).limit(10))




In [0]:
print(df_gold_final.count())